# **BASE CONDIVISA**

In [ ]:
!pip install xgboost -q

In [ ]:
import json, os, random, time, joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.ensemble import IsolationForest
from sklearn.metrics import auc, confusion_matrix, precision_recall_curve, roc_auc_score, roc_curve, classification_report
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import layers, models, optimizers, mixed_precision
from google.colab import drive

if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
AUTOTUNE = tf.data.AUTOTUNE

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

assert tuple(int(x) for x in tf.__version__.split(".")[:2]) >= (2, 11)

BASE_DIR = "/content/drive/MyDrive/ProgettoSicurezzaReti"
TFRECORD_DIR = os.path.join(BASE_DIR, "tfrecords")
CACHE_DIR = "/content/tfcache"

CNN_DIR = os.path.join(BASE_DIR, "cnn_model")
IF_DIR  = os.path.join(BASE_DIR, "if_model")
VAE_DIR = os.path.join(BASE_DIR, "vae_model")

CNN_MODEL_DIR, CNN_TRAIN_DIR, CNN_TEST_DIR = (os.path.join(CNN_DIR, s) for s in ["model", "train", "test"])
IF_MODEL_DIR, IF_TRAIN_DIR, IF_TEST_DIR = (os.path.join(IF_DIR, s) for s in ["model", "train", "test"])
VAE_MODEL_DIR, VAE_TRAIN_DIR, VAE_TEST_DIR = (os.path.join(VAE_DIR, s) for s in ["model", "train", "test"])

for d in [CNN_MODEL_DIR, CNN_TRAIN_DIR, CNN_TEST_DIR, IF_MODEL_DIR, IF_TRAIN_DIR, IF_TEST_DIR,
          VAE_MODEL_DIR, VAE_TRAIN_DIR, VAE_TEST_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

IMG_SIZE = 128
BATCH_SIZE = 32
MAX_EPOCHS = 50
PHASE1_EPOCHS = 10
PATIENCE = 7
VERBOSE = 2
WEIGHT_DECAY = 1e-3
DROPOUT = 0.30
LABEL_SMOOTHING = 0.01
INITIAL_LR = 1e-3
FINETUNE_LR = 1e-4
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.50
WARMUP_EPOCHS = 3

OBF_FEATURE_COLS = ["global_entropy", "block_entropy_mean", "block_entropy_std", "max_sect_entropy",
                     "avg_sect_entropy", "empty_sect_count", "rwx_sections", "wx_segments", "code_data_ratio"]

_train_counts = pd.read_csv(os.path.join(BASE_DIR, "dataset_train.csv"))["label"].value_counts()
N_GOODWARE = int(_train_counts.get(0, 0))
N_MALWARE = int(_train_counts.get(1, 0))
N_TOTAL = N_GOODWARE + N_MALWARE
W_GOODWARE = N_TOTAL / (2.0 * N_GOODWARE)
W_MALWARE = N_TOTAL / (2.0 * N_MALWARE)

GPUS = tf.config.list_physical_devices("GPU")
USE_AMP = len(GPUS) > 0
if USE_AMP:
    mixed_precision.set_global_policy("mixed_float16")
print(f"GPU: {'ON ' + GPUS[0].name if USE_AMP else 'non rilevata'} | mixed_float16: {'ON' if USE_AMP else 'OFF'}")

with open(os.path.join(TFRECORD_DIR, "meta.json")) as f:
    FEATURE_COLS = json.load(f)["feature_cols"]
CSV_DIM = len(FEATURE_COLS)
print(f"Feature CSV: {CSV_DIM} | Image: {IMG_SIZE}x{IMG_SIZE} | Batch: {BATCH_SIZE} | "
      f"W_goodware={W_GOODWARE:.3f} W_malware={W_MALWARE:.3f}")


def _save_fig(path):
    plt.tight_layout(); plt.savefig(path); plt.close()


def fit_obfuscation_labeler(df, contamination=0.15):
    scaler = StandardScaler().fit(df[OBF_FEATURE_COLS])
    X = scaler.transform(df[OBF_FEATURE_COLS])
    iso = IsolationForest(contamination=contamination, random_state=SEED, n_jobs=-1).fit(X)
    scores = -iso.score_samples(X)
    threshold = float(np.quantile(scores, 0.9))
    return {"scaler": scaler, "iso": iso, "threshold": threshold}


def apply_obfuscation_labels(df, labeler):
    X = labeler["scaler"].transform(df[OBF_FEATURE_COLS])
    scores = -labeler["iso"].score_samples(X)
    df = df.copy()
    df["obf_score"] = scores
    df["is_obfuscated"] = (scores > labeler["threshold"]).astype(int)
    return df


def fit_and_save_scalers():
    train_df = pd.read_csv(os.path.join(BASE_DIR, "dataset_train.csv"))
    val_df = pd.read_csv(os.path.join(BASE_DIR, "dataset_val.csv"))
    test_df = pd.read_csv(os.path.join(BASE_DIR, "dataset_test.csv"))
    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

    obf_labeler = fit_obfuscation_labeler(train_df)
    train_df = apply_obfuscation_labels(train_df, obf_labeler)
    val_df = apply_obfuscation_labels(val_df, obf_labeler)
    test_df = apply_obfuscation_labels(test_df, obf_labeler)

    n_obf = int(train_df["is_obfuscated"].sum())
    print(f"Offuscati train: {n_obf}/{len(train_df)} ({n_obf / len(train_df) * 100:.1f}%)")

    joblib.dump(obf_labeler, os.path.join(CNN_MODEL_DIR, "obf_labeler.pkl"))
    csv_scaler = StandardScaler().fit(train_df[FEATURE_COLS])
    joblib.dump(csv_scaler, os.path.join(CNN_MODEL_DIR, "csv_scaler.pkl"))
    with open(os.path.join(CNN_MODEL_DIR, "feature_cols.json"), "w") as f:
        json.dump(FEATURE_COLS, f)

    obf = {split: df.set_index("filename")["is_obfuscated"].to_dict()
           for split, df in [("train", train_df), ("val", val_df), ("test", test_df)]}
    return obf_labeler, csv_scaler, obf


def _parse_tfrecord(proto, csv_dim):
    feat_desc = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "csv_feat": tf.io.FixedLenFeature([csv_dim], tf.float32),
        "label": tf.io.FixedLenFeature([], tf.int64),
        "filename": tf.io.FixedLenFeature([], tf.string)
    }
    parsed = tf.io.parse_single_example(proto, feat_desc)
    image = tf.io.parse_tensor(parsed["image"], out_type=tf.float32)
    image.set_shape([IMG_SIZE, IMG_SIZE, 1])
    y_mal = tf.cast(parsed["label"], tf.float32)
    return image, parsed["csv_feat"], y_mal, parsed["filename"]


def make_dataset(split_name, obf_dict, csv_scaler, shuffle, cache=True, with_sample_weight=True):
    tfrecord_path = os.path.join(TFRECORD_DIR, f"{split_name}.tfrecord")
    keys = list(obf_dict.keys())
    values = [obf_dict[k] for k in keys]
    table = tf.lookup.StaticHashTable(
        tf.lookup.KeyValueTensorInitializer(tf.constant(keys, dtype=tf.string), tf.constant(values, dtype=tf.int32)),
        default_value=0
    )
    mean_ = tf.constant(csv_scaler.mean_, dtype=tf.float32)
    scale_ = tf.constant(csv_scaler.scale_, dtype=tf.float32)

    ds = tf.data.TFRecordDataset(tfrecord_path, num_parallel_reads=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=BATCH_SIZE * 64, reshuffle_each_iteration=True)
    ds = ds.map(lambda proto: _parse_tfrecord(proto, CSV_DIM), num_parallel_calls=AUTOTUNE)

    def to_keras_format(image, csv_feat, y_mal, filename):
        csv_feat_scaled = (csv_feat - mean_) / (scale_ + 1e-8)
        y_obf = tf.cast(table.lookup(filename), tf.float32)
        inputs = {"image": image, "csv_feat": csv_feat_scaled}
        targets = {"malware": y_mal, "obfuscated": y_obf}
        if with_sample_weight:
            w_mal = tf.where(tf.equal(y_mal, 1.0), W_MALWARE, W_GOODWARE)
            weights = {"malware": w_mal, "obfuscated": tf.ones_like(y_obf)}
            return inputs, targets, weights
        return inputs, targets

    ds = ds.map(to_keras_format, num_parallel_calls=AUTOTUNE)
    if cache:
        ds = ds.cache(os.path.join(CACHE_DIR, split_name))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(obf_dict), reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE, drop_remainder=True)
    return ds.prefetch(AUTOTUNE)


def errors_to_percentage(errors, err_min, err_max):
    return np.clip((errors - err_min) / (err_max - err_min + 1e-8), 0, 1) * 100


def load_scaler_and_labeler():
    csv_scaler_path = os.path.join(CNN_MODEL_DIR, "csv_scaler.pkl")
    obf_labeler_path = os.path.join(CNN_MODEL_DIR, "obf_labeler.pkl")
    if not os.path.isfile(csv_scaler_path) or not os.path.isfile(obf_labeler_path):
        raise FileNotFoundError("Scaler/labeler non trovati. Esegui prima la cella CNN.")
    return joblib.load(csv_scaler_path), joblib.load(obf_labeler_path)


def load_test_dataset(csv_scaler, obf_labeler, with_sample_weight=False):
    test_df = pd.read_csv(os.path.join(BASE_DIR, "dataset_test.csv"))
    test_df = apply_obfuscation_labels(test_df, obf_labeler)
    obf_test = test_df.set_index("filename")["is_obfuscated"].to_dict()
    return make_dataset("test", obf_test, csv_scaler, shuffle=False, cache=True, with_sample_weight=with_sample_weight)


def read_tfrecord_filenames(split_name):
    filenames = []
    path = os.path.join(TFRECORD_DIR, f"{split_name}.tfrecord")
    for proto in tf.data.TFRecordDataset(path):
        p = tf.io.parse_single_example(proto, {"filename": tf.io.FixedLenFeature([], tf.string)})
        filenames.append(p["filename"].numpy().decode())
    return filenames


def save_unsupervised_predictions(pct, output_dir, csv_name):
    filenames = read_tfrecord_filenames("test")
    n = min(len(filenames), len(pct))
    csv_out = os.path.join(output_dir, csv_name)
    pd.DataFrame({"filename": filenames[:n], "obf_pct_unsupervised": pct[:n]}).to_csv(csv_out, index=False)
    return csv_out

Mounted at /content/drive
GPU: ON /physical_device:GPU:0 | mixed_float16: ON
Feature CSV: 560 | Image: 128x128 | Batch: 32 | W_goodware=0.678 W_malware=1.904


# **CNN**

In [ ]:
EFFICIENTNET_WEIGHTS_PATH = os.path.join(CNN_MODEL_DIR, "efficientnetb0_notop.h5")


def build_csv_branch(csv_in_dim, dropout=0.30):
    inp = layers.Input(shape=(csv_in_dim,), name="csv_feat_input")
    x = layers.Dense(256, activation="swish")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    residual = x
    x = layers.Dense(256, activation="swish")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Add()([x, residual])
    x = layers.Dense(128, activation="swish")(x)
    x = layers.BatchNormalization()(x)
    return models.Model(inp, x, name="CSVResidualMLP")


def build_malware_cnn(csv_in_dim):
    image_in = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name="image")
    csv_in = layers.Input(shape=(csv_in_dim,), name="csv_feat")

    x_img = layers.RandomTranslation(0.05, 0.05, fill_mode="constant", fill_value=0.0)(image_in)
    x_img = layers.RandomContrast(0.10)(x_img)
    x_img = layers.GaussianNoise(0.02)(x_img)
    x_img = layers.Concatenate(name="gray_to_rgb")([x_img, x_img, x_img])

    backbone = tf.keras.applications.EfficientNetB0(
        include_top=False, weights=None,
        input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg", name="EfficientNetB0"
    )
    if os.path.isfile(EFFICIENTNET_WEIGHTS_PATH):
        backbone.load_weights(EFFICIENTNET_WEIGHTS_PATH)
        print(f"Pesi ImageNet caricati da {EFFICIENTNET_WEIGHTS_PATH}")
    else:
        print(f"ATTENZIONE: {EFFICIENTNET_WEIGHTS_PATH} non trovato — backbone non pre-addestrato (pesi random).")

    backbone.trainable = False
    x_img = backbone(x_img)

    x_img = layers.BatchNormalization(name="image_projection_bn")(x_img)
    x_img = layers.Dense(256, activation="swish", name="image_projection")(x_img)
    x_img = layers.Dropout(DROPOUT)(x_img)

    x_csv = build_csv_branch(csv_in_dim, dropout=DROPOUT)(csv_in)
    x_csv = layers.Dense(256, activation="swish", name="csv_projection")(x_csv)
    x_csv = layers.BatchNormalization(name="csv_projection_bn")(x_csv)

    gate = layers.Dense(256, activation="sigmoid", name="feature_gate")(layers.Concatenate()([x_img, x_csv]))
    x_img_gated = layers.Multiply()([x_img, gate])
    x_csv_gated = layers.Multiply()([x_csv, 1.0 - gate])

    shared = layers.Concatenate(name="multimodal_fusion")([x_img_gated, x_csv_gated])
    shared = layers.Dense(256, activation="swish", name="fusion_dense")(shared)
    shared = layers.BatchNormalization(name="fusion_bn")(shared)
    shared = layers.Dropout(DROPOUT)(shared)

    mal = layers.Dense(128, activation="swish")(shared)
    mal = layers.BatchNormalization()(mal)
    mal = layers.Dropout(0.20)(mal)
    out_mal = layers.Dense(1, activation="sigmoid", dtype="float32", name="malware")(mal)

    obf = layers.Dense(128, activation="swish")(shared)
    obf = layers.BatchNormalization()(obf)
    obf = layers.Dropout(0.20)(obf)
    out_obf = layers.Dense(1, activation="sigmoid", dtype="float32", name="obfuscated")(obf)

    model = models.Model(inputs=[image_in, csv_in],
                          outputs={"malware": out_mal, "obfuscated": out_obf},
                          name="MalwareEfficientNetB0")
    return model, backbone


def load_trained_cnn(model_path):
    import zipfile
    cnn_model, _ = build_malware_cnn(CSV_DIM)
    with zipfile.ZipFile(model_path, "r") as z:
        weights_name = next((n for n in z.namelist() if n.endswith(".weights.h5")), None)
        if weights_name is None:
            raise FileNotFoundError(f"Nessun .weights.h5 in {model_path}")
        extract_dir = "/tmp/cnn_extract"
        z.extract(weights_name, extract_dir)
    cnn_model.load_weights(os.path.join(extract_dir, weights_name))
    return cnn_model


def extract_fusion_embeddings(cnn_model, dataset):
    embedder = models.Model(cnn_model.input, cnn_model.get_layer("fusion_bn").output)
    embs, y_mal_list, y_obf_list = [], [], []
    for inputs, targets in dataset:
        embs.append(tf.cast(embedder(inputs, training=False), tf.float32).numpy())
        y_mal_list.append(targets["malware"].numpy())
        y_obf_list.append(targets["obfuscated"].numpy())
    return np.concatenate(embs), np.concatenate(y_mal_list), np.concatenate(y_obf_list)


def get_predictions(model, dataset):
    labels_mal, labels_obf, probs_mal, probs_obf = [], [], [], []
    for inputs, targets in dataset:
        preds = model(inputs, training=False)
        labels_mal.append(targets["malware"].numpy())
        labels_obf.append(targets["obfuscated"].numpy())
        probs_mal.append(np.asarray(preds["malware"]).flatten())
        probs_obf.append(np.asarray(preds["obfuscated"]).flatten())
    return {
        "labels_mal": np.concatenate(labels_mal).flatten(),
        "probs_mal": np.concatenate(probs_mal),
        "labels_obf": np.concatenate(labels_obf).flatten(),
        "probs_obf": np.concatenate(probs_obf)
    }


def print_classification_report(res):
    print("\n--- MALWARE ---")
    print(classification_report(res["labels_mal"], (res["probs_mal"] > 0.5).astype(int),
                                 target_names=["Goodware", "Malware"]))
    print(f"ROC-AUC: {roc_auc_score(res['labels_mal'], res['probs_mal']):.4f}")
    print("\n--- OFFUSCAMENTO ---")
    print(classification_report(res["labels_obf"], (res["probs_obf"] > 0.5).astype(int),
                                 target_names=["Non offuscato", "Offuscato"]))
    print(f"ROC-AUC: {roc_auc_score(res['labels_obf'], res['probs_obf']):.4f}")


def plot_confusion_matrix(labels, preds, title, out_path, cmap="Blues"):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap)
    plt.title(title); plt.xlabel("Predetto"); plt.ylabel("Reale")
    _save_fig(out_path)


def save_plots(history, res, output_dir):
    pm, lm = np.array(res["probs_mal"]), np.array(res["labels_mal"])
    po, lo = np.array(res["probs_obf"]), np.array(res["labels_obf"])

    if history is not None and "loss" in history.history:
        plt.figure(figsize=(8, 4))
        plt.plot(history.history["loss"], label="Train")
        plt.plot(history.history["val_loss"], label="Val")
        plt.title("Loss per Epoca"); plt.xlabel("Epoca"); plt.ylabel("Loss"); plt.legend()
        _save_fig(os.path.join(output_dir, "loss_curve.png"))

    plot_confusion_matrix(lm, (pm > 0.5).astype(int), "Confusion Matrix - Malware",
                           os.path.join(output_dir, "confusion_matrix_malware.png"))
    plot_confusion_matrix(lo, (po > 0.5).astype(int), "Confusion Matrix - Offuscato",
                           os.path.join(output_dir, "confusion_matrix_obfuscation.png"), cmap="Oranges")

    df_res = pd.DataFrame({"Confidenza": pm, "Classe": lm}).replace({"Classe": {0: "Goodware", 1: "Malware"}})
    plt.figure(figsize=(8, 6))
    sns.boxplot(x="Classe", y="Confidenza", data=df_res, hue="Classe",
                palette={"Goodware": "blue", "Malware": "red"}, legend=False)
    plt.axhline(0.5, color="black", linestyle="--")
    _save_fig(os.path.join(output_dir, "confidence_boxplot_malware.png"))

    for name, labels, probs, fname in [("Malware", lm, pm, "malware"), ("Offuscamento", lo, po, "obfuscation")]:
        fpr, tpr, _ = roc_curve(labels, probs)
        plt.figure(figsize=(6, 5))
        plt.plot(fpr, tpr, label=f"AUC={auc(fpr, tpr):.4f}")
        plt.plot([0, 1], [0, 1], "k--")
        plt.title(f"ROC - {name}"); plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.legend()
        _save_fig(os.path.join(output_dir, f"roc_curve_{fname}.png"))

        prec, rec, _ = precision_recall_curve(labels, probs)
        plt.figure(figsize=(6, 5))
        plt.plot(rec, prec)
        plt.title(f"PR Curve - {name}"); plt.xlabel("Recall"); plt.ylabel("Precision")
        _save_fig(os.path.join(output_dir, f"pr_curve_{fname}.png"))


def evaluate_cnn_on_test(test_ds_no_weight):
    cnn_model = load_trained_cnn(os.path.join(CNN_MODEL_DIR, "best_model.keras"))
    res = get_predictions(cnn_model, test_ds_no_weight)
    print_classification_report(res)
    save_plots(None, res, output_dir=CNN_TEST_DIR)
    return cnn_model, res


class LinearWarmupCallback(tf.keras.callbacks.Callback):
    def __init__(self, target_lr, warmup_epochs):
        super().__init__()
        self.target_lr = target_lr
        self.warmup_epochs = max(int(warmup_epochs), 0)

    def on_epoch_begin(self, epoch, logs=None):
        if self.warmup_epochs == 0 or epoch >= self.warmup_epochs:
            return
        self.model.optimizer.learning_rate.assign(self.target_lr * (epoch + 1) / self.warmup_epochs)


def compile_cnn(model, learning_rate):
    optimizer = optimizers.AdamW(learning_rate=learning_rate, weight_decay=WEIGHT_DECAY,
                                  beta_1=0.9, beta_2=0.999, clipnorm=1.0)
    if USE_AMP:
        optimizer = mixed_precision.LossScaleOptimizer(optimizer)

    model.compile(
        optimizer=optimizer,
        loss={
            "malware": tf.keras.losses.BinaryFocalCrossentropy(
                gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA, label_smoothing=LABEL_SMOOTHING
            ),
            "obfuscated": tf.keras.losses.BinaryCrossentropy(),
        },
        metrics={
            "malware": [tf.keras.metrics.AUC(name="auc"), tf.keras.metrics.BinaryAccuracy(name="acc")],
            "obfuscated": [tf.keras.metrics.AUC(name="auc"), tf.keras.metrics.BinaryAccuracy(name="acc")],
        }
    )
    return model


def train_cnn(train_ds, val_ds):
    model, backbone = build_malware_cnn(CSV_DIM)
    best_model_path = os.path.join(CNN_MODEL_DIR, "best_model.keras")
    last_model_path = os.path.join(CNN_MODEL_DIR, "last_model.keras")
    state_path = os.path.join(CNN_MODEL_DIR, "train_state.json")

    checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(best_model_path, monitor="val_malware_loss", mode="min",
                                                       save_best_only=True, verbose=0)
    last_checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(last_model_path, save_best_only=False, verbose=0)
    early_stopping_cb = tf.keras.callbacks.EarlyStopping(monitor="val_malware_loss", mode="min", patience=PATIENCE,
                                                         restore_best_weights=True, verbose=1)
    reduce_lr_cb = tf.keras.callbacks.ReduceLROnPlateau(monitor="val_malware_loss", mode="min", factor=0.5,
                                                        patience=max(PATIENCE // 2, 2), min_lr=1e-6, verbose=1)

    class StateSaverCallback(tf.keras.callbacks.Callback):
        def __init__(self, phase):
            super().__init__()
            self.phase = phase
        def on_epoch_end(self, epoch, logs=None):
            with open(state_path, "w") as f:
                json.dump({"phase": self.phase, "last_completed_epoch": epoch}, f)

    # --- Resume state ---
    start_phase = 1
    start_epoch_p1 = 0
    start_epoch_p2 = PHASE1_EPOCHS
    history_p1_dict = {}

    if os.path.exists(state_path) and os.path.exists(last_model_path):
        with open(state_path) as f:
            state = json.load(f)
        model.load_weights(last_model_path)
        if state["phase"] == 1:
            start_epoch_p1 = state["last_completed_epoch"] + 1
            start_phase = 1
        else:
            start_phase = 2
            start_epoch_p2 = state["last_completed_epoch"] + 1
        print(f"Resume da fase {state['phase']}, epoca {state['last_completed_epoch'] + 1}")

    # --- FASE 1 ---
    if start_phase == 1 and start_epoch_p1 < PHASE1_EPOCHS:
        print(f"\n--- FASE 1: Addestramento Head | LR={INITIAL_LR:.1e} | Epoche: {PHASE1_EPOCHS} ---")
        backbone.trainable = False
        model = compile_cnn(model, INITIAL_LR)
        callbacks_p1 = [LinearWarmupCallback(target_lr=INITIAL_LR, warmup_epochs=WARMUP_EPOCHS),
                        checkpoint_cb, last_checkpoint_cb, StateSaverCallback(phase=1)]
        history_p1 = model.fit(train_ds, validation_data=val_ds, initial_epoch=start_epoch_p1,
                               epochs=PHASE1_EPOCHS, callbacks=callbacks_p1, verbose=VERBOSE)
        history_p1_dict = history_p1.history
    else:
        print("FASE 1 già completata, salto.")

    # --- FASE 2 ---
    print(f"\n--- FASE 2: Fine-tuning Globale | LR={FINETUNE_LR:.1e} | Epoche rimanenti: {MAX_EPOCHS - PHASE1_EPOCHS} ---")
    if start_phase == 1 and os.path.exists(best_model_path):
        model.load_weights(best_model_path)
    backbone.trainable = True
    model = compile_cnn(model, FINETUNE_LR)
    callbacks_p2 = [checkpoint_cb, last_checkpoint_cb, StateSaverCallback(phase=2),
                    early_stopping_cb, reduce_lr_cb]
    history_p2 = model.fit(train_ds, validation_data=val_ds,
                           initial_epoch=start_epoch_p2, epochs=MAX_EPOCHS,
                           callbacks=callbacks_p2, verbose=VERBOSE)

    for key in history_p2.history.keys():
        history_p1_dict.setdefault(key, []).extend(history_p2.history.get(key, []))

    best_idx = int(np.argmin(history_p1_dict["val_malware_loss"]))
    print(f"\nBest epoch {best_idx + 1}: val_malware_loss={history_p1_dict['val_malware_loss'][best_idx]:.4f}")
    model.load_weights(best_model_path)

    if os.path.exists(state_path):
        os.remove(state_path)

    class H: pass
    h = H(); h.history = history_p1_dict
    return model, h


def main_train_cnn():
    t_start = time.time()
    _, csv_scaler, obf_dicts = fit_and_save_scalers()
    train_ds = make_dataset("train", obf_dicts["train"], csv_scaler, shuffle=True, with_sample_weight=True)
    val_ds = make_dataset("val", obf_dicts["val"], csv_scaler, shuffle=False, with_sample_weight=True)
    model, history = train_cnn(train_ds, val_ds)
    val_ds_eval = make_dataset("val", obf_dicts["val"], csv_scaler, shuffle=False, with_sample_weight=False)
    res = get_predictions(model, val_ds_eval)
    print_classification_report(res)
    save_plots(history, res, output_dir=CNN_TRAIN_DIR)
    print(f"Training CNN completato in {(time.time() - t_start) / 60:.1f} min")
    return model


def main_test_cnn():
    t_start = time.time()
    csv_scaler, obf_labeler = load_scaler_and_labeler()
    test_ds = load_test_dataset(csv_scaler, obf_labeler)
    cnn_model, res = evaluate_cnn_on_test(test_ds)
    print(f"Test CNN completato in {(time.time() - t_start) / 60:.1f} min")
    return cnn_model


import shutil
shutil.rmtree(CACHE_DIR, ignore_errors=True)
os.makedirs(CACHE_DIR, exist_ok=True)

#cnn_model = main_train_cnn()
cnn_model = main_test_cnn()

Pesi ImageNet caricati da /content/drive/MyDrive/ProgettoSicurezzaReti/cnn_model/model/efficientnetb0_notop.h5


# **ISOLATION FOREST**

In [ ]:
IF_N_ESTIMATORS = 300
IF_CONTAMINATION = 0.15
import time

def plot_if_results(scores, y_obf, pct, output_dir):
    plt.figure(figsize=(8, 5))
    sns.histplot(scores[y_obf == 0], color="blue", label="Non offuscato", kde=True, stat="density", alpha=0.4)
    sns.histplot(scores[y_obf == 1], color="red", label="Offuscato", kde=True, stat="density", alpha=0.4)
    plt.title("Distribuzione Anomaly Score — Isolation Forest"); plt.legend()
    _save_fig(os.path.join(output_dir, "if_score_distribution.png"))

    df_score = pd.DataFrame({"Score": scores, "Classe": y_obf}).replace({"Classe": {0: "Non offuscato", 1: "Offuscato"}})
    plt.figure(figsize=(7, 5))
    sns.boxplot(x="Classe", y="Score", data=df_score, hue="Classe",
                palette={"Non offuscato": "blue", "Offuscato": "red"}, legend=False)
    plt.title("Anomaly Score per Classe")
    _save_fig(os.path.join(output_dir, "if_score_boxplot.png"))

    fpr, tpr, _ = roc_curve(y_obf, scores)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC={auc(fpr, tpr):.4f}"); plt.plot([0, 1], [0, 1], "k--")
    plt.title("ROC — Isolation Forest"); plt.legend()
    _save_fig(os.path.join(output_dir, "if_roc_curve.png"))

    prec, rec, _ = precision_recall_curve(y_obf, scores)
    plt.figure(figsize=(6, 5))
    plt.plot(rec, prec); plt.title("PR Curve — Isolation Forest")
    _save_fig(os.path.join(output_dir, "if_pr_curve.png"))

    plt.figure(figsize=(7, 5))
    plt.scatter(scores, pct, c=y_obf, cmap="coolwarm", alpha=0.5, s=10)
    plt.title("Score vs Percentuale Calibrata"); plt.colorbar(label="is_obfuscated")
    _save_fig(os.path.join(output_dir, "if_score_vs_pct.png"))


def train_embedding_isolation_forest(train_emb):
    t0 = time.time()
    iso = IsolationForest(n_estimators=IF_N_ESTIMATORS, contamination=IF_CONTAMINATION,
                           random_state=SEED, n_jobs=-1).fit(train_emb)
    print(f"Isolation Forest fit completato in {time.time() - t0:.1f}s")
    return iso


def compute_if_anomaly_scores(iso, embeddings):
    return -iso.score_samples(embeddings)


def main_train_if():
    t_start = time.time()
    _, csv_scaler, obf_dicts = fit_and_save_scalers()
    train_ds = make_dataset("train", obf_dicts["train"], csv_scaler, shuffle=True, with_sample_weight=False)

    best_model_path = os.path.join(CNN_MODEL_DIR, "best_model.keras")
    if not os.path.isfile(best_model_path):
        raise FileNotFoundError(f"CNN non trovata in {best_model_path}. Esegui prima la cella CNN.")
    cnn_model = load_trained_cnn(best_model_path)

    train_emb, _, y_obf_train = extract_fusion_embeddings(cnn_model, train_ds)
    iso_model = train_embedding_isolation_forest(train_emb)
    joblib.dump(iso_model, os.path.join(IF_MODEL_DIR, "iso_embedding.pkl"))

    train_scores = compute_if_anomaly_scores(iso_model, train_emb)
    score_min, score_max = float(train_scores.min()), float(train_scores.max())
    with open(os.path.join(IF_MODEL_DIR, "if_calibration.json"), "w") as f:
        json.dump({"score_min": score_min, "score_max": score_max, "embedding_source": "fusion_bn",
                    "n_estimators": IF_N_ESTIMATORS, "contamination": IF_CONTAMINATION}, f)

    train_pct = errors_to_percentage(train_scores, score_min, score_max)
    plot_if_results(train_scores, y_obf_train, train_pct, IF_TRAIN_DIR)
    print(f"IF ROC-AUC (train): {roc_auc_score(y_obf_train, train_scores):.4f}")
    print(f"Training Isolation Forest completato in {(time.time() - t_start) / 60:.1f} min")
    return iso_model


def main_test_if():
    t_start = time.time()
    csv_scaler, obf_labeler = load_scaler_and_labeler()
    test_ds_eval = load_test_dataset(csv_scaler, obf_labeler)
    cnn_model, _ = evaluate_cnn_on_test(test_ds_eval)

    iso_model = joblib.load(os.path.join(IF_MODEL_DIR, "iso_embedding.pkl"))
    with open(os.path.join(IF_MODEL_DIR, "if_calibration.json")) as f:
        calib = json.load(f)

    test_emb, _, y_obf_test = extract_fusion_embeddings(cnn_model, test_ds_eval)
    test_scores = compute_if_anomaly_scores(iso_model, test_emb)
    test_pct = errors_to_percentage(test_scores, calib["score_min"], calib["score_max"])

    print(f"IF ROC-AUC (test): {roc_auc_score(y_obf_test, test_scores):.4f}")
    plot_if_results(test_scores, y_obf_test, test_pct, IF_TEST_DIR)
    csv_out = save_unsupervised_predictions(test_pct, IF_TEST_DIR, "obfuscation_percentage_test_if.csv")
    print(f"Test Isolation Forest completato in {(time.time() - t_start) / 60:.1f} min | CSV: {csv_out}")

# --- Funzioni minime dalla cella CNN, necessarie a Isolation Forest ---

EFFICIENTNET_WEIGHTS_PATH = os.path.join(CNN_MODEL_DIR, "efficientnetb0_notop.h5")


def build_csv_branch(csv_in_dim, dropout=0.30):
    inp = layers.Input(shape=(csv_in_dim,), name="csv_feat_input")
    x = layers.Dense(256, activation="swish")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    residual = x
    x = layers.Dense(256, activation="swish")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Add()([x, residual])
    x = layers.Dense(128, activation="swish")(x)
    x = layers.BatchNormalization()(x)
    return models.Model(inp, x, name="CSVResidualMLP")


def build_malware_cnn(csv_in_dim):
    image_in = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name="image")
    csv_in = layers.Input(shape=(csv_in_dim,), name="csv_feat")

    x_img = layers.RandomTranslation(0.05, 0.05, fill_mode="constant", fill_value=0.0)(image_in)
    x_img = layers.RandomContrast(0.10)(x_img)
    x_img = layers.GaussianNoise(0.02)(x_img)
    x_img = layers.Concatenate(name="gray_to_rgb")([x_img, x_img, x_img])

    backbone = tf.keras.applications.EfficientNetB0(
        include_top=False, weights=None,
        input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg", name="EfficientNetB0"
    )
    if os.path.isfile(EFFICIENTNET_WEIGHTS_PATH):
        backbone.load_weights(EFFICIENTNET_WEIGHTS_PATH)
        print(f"Pesi ImageNet caricati da {EFFICIENTNET_WEIGHTS_PATH}")
    else:
        print(f"ATTENZIONE: {EFFICIENTNET_WEIGHTS_PATH} non trovato — backbone non pre-addestrato (pesi random).")

    backbone.trainable = False
    x_img = backbone(x_img)

    x_img = layers.BatchNormalization(name="image_projection_bn")(x_img)
    x_img = layers.Dense(256, activation="swish", name="image_projection")(x_img)
    x_img = layers.Dropout(DROPOUT)(x_img)

    x_csv = build_csv_branch(csv_in_dim, dropout=DROPOUT)(csv_in)
    x_csv = layers.Dense(256, activation="swish", name="csv_projection")(x_csv)
    x_csv = layers.BatchNormalization(name="csv_projection_bn")(x_csv)

    gate = layers.Dense(256, activation="sigmoid", name="feature_gate")(layers.Concatenate()([x_img, x_csv]))
    x_img_gated = layers.Multiply()([x_img, gate])
    x_csv_gated = layers.Multiply()([x_csv, 1.0 - gate])

    shared = layers.Concatenate(name="multimodal_fusion")([x_img_gated, x_csv_gated])
    shared = layers.Dense(256, activation="swish", name="fusion_dense")(shared)
    shared = layers.BatchNormalization(name="fusion_bn")(shared)
    shared = layers.Dropout(DROPOUT)(shared)

    mal = layers.Dense(128, activation="swish")(shared)
    mal = layers.BatchNormalization()(mal)
    mal = layers.Dropout(0.20)(mal)
    out_mal = layers.Dense(1, activation="sigmoid", dtype="float32", name="malware")(mal)

    obf = layers.Dense(128, activation="swish")(shared)
    obf = layers.BatchNormalization()(obf)
    obf = layers.Dropout(0.20)(obf)
    out_obf = layers.Dense(1, activation="sigmoid", dtype="float32", name="obfuscated")(obf)

    model = models.Model(inputs=[image_in, csv_in],
                          outputs={"malware": out_mal, "obfuscated": out_obf},
                          name="MalwareEfficientNetB0")
    return model, backbone


def load_trained_cnn(model_path):
    import zipfile
    cnn_model, _ = build_malware_cnn(CSV_DIM)
    with zipfile.ZipFile(model_path, "r") as z:
        weights_name = next((n for n in z.namelist() if n.endswith(".weights.h5")), None)
        if weights_name is None:
            raise FileNotFoundError(f"Nessun .weights.h5 in {model_path}")
        extract_dir = "/tmp/cnn_extract"
        z.extract(weights_name, extract_dir)
    cnn_model.load_weights(os.path.join(extract_dir, weights_name))
    return cnn_model


def extract_fusion_embeddings(cnn_model, dataset):
    embedder = models.Model(cnn_model.input, cnn_model.get_layer("fusion_bn").output)
    embs, y_mal_list, y_obf_list = [], [], []
    for inputs, targets in dataset:
        embs.append(tf.cast(embedder(inputs, training=False), tf.float32).numpy())
        y_mal_list.append(targets["malware"].numpy())
        y_obf_list.append(targets["obfuscated"].numpy())
    return np.concatenate(embs), np.concatenate(y_mal_list), np.concatenate(y_obf_list)


def get_predictions(model, dataset):
    labels_mal, labels_obf, probs_mal, probs_obf = [], [], [], []
    for inputs, targets in dataset:
        preds = model(inputs, training=False)
        labels_mal.append(targets["malware"].numpy())
        labels_obf.append(targets["obfuscated"].numpy())
        probs_mal.append(np.asarray(preds["malware"]).flatten())
        probs_obf.append(np.asarray(preds["obfuscated"]).flatten())
    return {
        "labels_mal": np.concatenate(labels_mal).flatten(),
        "probs_mal": np.concatenate(probs_mal),
        "labels_obf": np.concatenate(labels_obf).flatten(),
        "probs_obf": np.concatenate(probs_obf)
    }


def print_classification_report(res):
    print("\n--- MALWARE ---")
    print(classification_report(res["labels_mal"], (res["probs_mal"] > 0.5).astype(int),
                                 target_names=["Goodware", "Malware"]))
    print(f"ROC-AUC: {roc_auc_score(res['labels_mal'], res['probs_mal']):.4f}")
    print("\n--- OFFUSCAMENTO ---")
    print(classification_report(res["labels_obf"], (res["probs_obf"] > 0.5).astype(int),
                                 target_names=["Non offuscato", "Offuscato"]))
    print(f"ROC-AUC: {roc_auc_score(res['labels_obf'], res['probs_obf']):.4f}")


def plot_confusion_matrix(labels, preds, title, out_path, cmap="Blues"):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap)
    plt.title(title); plt.xlabel("Predetto"); plt.ylabel("Reale")
    _save_fig(out_path)


def save_plots(history, res, output_dir):
    pm, lm = np.array(res["probs_mal"]), np.array(res["labels_mal"])
    po, lo = np.array(res["probs_obf"]), np.array(res["labels_obf"])

    if history is not None and "loss" in history.history:
        plt.figure(figsize=(8, 4))
        plt.plot(history.history["loss"], label="Train")
        plt.plot(history.history["val_loss"], label="Val")
        plt.title("Loss per Epoca"); plt.xlabel("Epoca"); plt.ylabel("Loss"); plt.legend()
        _save_fig(os.path.join(output_dir, "loss_curve.png"))

    plot_confusion_matrix(lm, (pm > 0.5).astype(int), "Confusion Matrix - Malware",
                           os.path.join(output_dir, "confusion_matrix_malware.png"))
    plot_confusion_matrix(lo, (po > 0.5).astype(int), "Confusion Matrix - Offuscato",
                           os.path.join(output_dir, "confusion_matrix_obfuscation.png"), cmap="Oranges")

    df_res = pd.DataFrame({"Confidenza": pm, "Classe": lm}).replace({"Classe": {0: "Goodware", 1: "Malware"}})
    plt.figure(figsize=(8, 6))
    sns.boxplot(x="Classe", y="Confidenza", data=df_res, hue="Classe",
                palette={"Goodware": "blue", "Malware": "red"}, legend=False)
    plt.axhline(0.5, color="black", linestyle="--")
    _save_fig(os.path.join(output_dir, "confidence_boxplot_malware.png"))

    for name, labels, probs, fname in [("Malware", lm, pm, "malware"), ("Offuscamento", lo, po, "obfuscation")]:
        fpr, tpr, _ = roc_curve(labels, probs)
        plt.figure(figsize=(6, 5))
        plt.plot(fpr, tpr, label=f"AUC={auc(fpr, tpr):.4f}")
        plt.plot([0, 1], [0, 1], "k--")
        plt.title(f"ROC - {name}"); plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.legend()
        _save_fig(os.path.join(output_dir, f"roc_curve_{fname}.png"))

        prec, rec, _ = precision_recall_curve(labels, probs)
        plt.figure(figsize=(6, 5))
        plt.plot(rec, prec)
        plt.title(f"PR Curve - {name}"); plt.xlabel("Recall"); plt.ylabel("Precision")
        _save_fig(os.path.join(output_dir, f"pr_curve_{fname}.png"))


def evaluate_cnn_on_test(test_ds_no_weight):
    cnn_model = load_trained_cnn(os.path.join(CNN_MODEL_DIR, "best_model.keras"))
    res = get_predictions(cnn_model, test_ds_no_weight)
    print_classification_report(res)
    save_plots(None, res, output_dir=CNN_TEST_DIR)
    return cnn_model, res

# --- Fine funzioni riportate dalla cella CNN ---


iso_model = main_train_if()
main_test_if()

Train: 36047 | Val: 7725 | Test: 7723
Offuscati train: 3571/36047 (9.9%)
Pesi ImageNet caricati da /content/drive/MyDrive/ProgettoSicurezzaReti/cnn_model/model/efficientnetb0_notop.h5
Isolation Forest fit completato in 2.0s
IF ROC-AUC (train): 0.9346
Training Isolation Forest completato in 10.1 min
Pesi ImageNet caricati da /content/drive/MyDrive/ProgettoSicurezzaReti/cnn_model/model/efficientnetb0_notop.h5

--- MALWARE ---
              precision    recall  f1-score   support

    Goodware       1.00      1.00      1.00      5684
     Malware       1.00      1.00      1.00      2028

    accuracy                           1.00      7712
   macro avg       1.00      1.00      1.00      7712
weighted avg       1.00      1.00      1.00      7712

ROC-AUC: 1.0000

--- OFFUSCAMENTO ---
               precision    recall  f1-score   support

Non offuscato       1.00      0.99      1.00      6961
    Offuscato       0.95      0.97      0.96       751

     accuracy                           

# **VARIENTIONAL ENCODER**

In [ ]:
VAE_LATENT_DIM = 32
VAE_EPOCHS = 100
VAE_PATIENCE = 10
VAE_BATCH_SIZE = 64
VAE_LR = 1e-3
KL_WEIGHT = 0.001


class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        eps = tf.random.normal(shape=tf.shape(z_mean), dtype=z_mean.dtype)
        return z_mean + tf.exp(0.5 * z_log_var) * eps


def build_vae(input_dim, latent_dim=VAE_LATENT_DIM):
    enc_in = layers.Input(shape=(input_dim,), dtype="float32")
    x = layers.Dense(128, activation="swish", dtype="float32")(enc_in)
    x = layers.BatchNormalization(dtype="float32")(x)
    x = layers.Dense(64, activation="swish", dtype="float32")(x)
    x = layers.BatchNormalization(dtype="float32")(x)
    z_mean = layers.Dense(latent_dim, name="z_mean", dtype="float32")(x)
    z_log_var = layers.Dense(latent_dim, name="z_log_var", dtype="float32")(x)
    z = Sampling(dtype="float32")([z_mean, z_log_var])
    encoder = models.Model(enc_in, [z_mean, z_log_var, z], name="VAE_Encoder")

    dec_in = layers.Input(shape=(latent_dim,), dtype="float32")
    y = layers.Dense(64, activation="swish", dtype="float32")(dec_in)
    y = layers.BatchNormalization(dtype="float32")(y)
    y = layers.Dense(128, activation="swish", dtype="float32")(y)
    y = layers.BatchNormalization(dtype="float32")(y)
    out = layers.Dense(input_dim, dtype="float32")(y)
    decoder = models.Model(dec_in, out, name="VAE_Decoder")
    return encoder, decoder


class VAE(models.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.recon_loss_tracker = tf.keras.metrics.Mean(name="recon_loss")
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.recon_loss_tracker, self.kl_loss_tracker]

    def _compute_losses(self, data, training):
        z_mean, z_log_var, z = self.encoder(data, training=training)
        recon = self.decoder(z, training=training)
        recon_loss = tf.reduce_mean(tf.reduce_sum(tf.square(data - recon), axis=1))
        kl_loss = -0.5 * tf.reduce_mean(tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1))
        return recon_loss + KL_WEIGHT * kl_loss, recon_loss, kl_loss

    def train_step(self, data):
        with tf.GradientTape() as tape:
            total_loss, recon_loss, kl_loss = self._compute_losses(data, training=True)
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        total_loss, recon_loss, kl_loss = self._compute_losses(data, training=False)
        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {m.name: m.result() for m in self.metrics}

    def call(self, data):
        _, _, z = self.encoder(data)
        return self.decoder(z)


def compute_vae_reconstruction_errors(vae, embeddings, batch_size=512):
    errors = []
    embeddings = embeddings.astype(np.float32)
    for i in range(0, len(embeddings), batch_size):
        batch = embeddings[i:i + batch_size]
        z_mean, _, _ = vae.encoder(batch, training=False)
        recon = vae.decoder(z_mean, training=False)
        errors.append(np.mean(np.square(batch - recon.numpy()), axis=1))
    return np.concatenate(errors)


def train_vae_on_embeddings(train_emb, val_emb):
    train_emb, val_emb = train_emb.astype(np.float32), val_emb.astype(np.float32)
    encoder, decoder = build_vae(train_emb.shape[1], VAE_LATENT_DIM)
    vae = VAE(encoder, decoder)
    vae.compile(optimizer=optimizers.Adam(learning_rate=VAE_LR))

    train_ds = tf.data.Dataset.from_tensor_slices(train_emb).shuffle(len(train_emb)).batch(VAE_BATCH_SIZE).prefetch(AUTOTUNE)
    val_ds = tf.data.Dataset.from_tensor_slices(val_emb).batch(VAE_BATCH_SIZE).prefetch(AUTOTUNE)

    early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=VAE_PATIENCE,
                                                   restore_best_weights=True, verbose=1)
    history = vae.fit(train_ds, validation_data=val_ds, epochs=VAE_EPOCHS, callbacks=[early_stop], verbose=VERBOSE)
    print(f"VAE best val_loss: {min(history.history['val_loss']):.4f}")
    return vae, history.history["loss"], history.history["val_loss"]


def plot_vae_results(train_losses, val_losses, errors, y_obf, pct, output_dir):
    if train_losses and val_losses:
        plt.figure(figsize=(8, 4))
        plt.plot(train_losses, label="Train"); plt.plot(val_losses, label="Val")
        plt.title("VAE — Loss per Epoca"); plt.legend()
        _save_fig(os.path.join(output_dir, "vae_loss_curve.png"))

    plt.figure(figsize=(8, 5))
    sns.histplot(errors[y_obf == 0], color="blue", label="Non offuscato", kde=True, stat="density", alpha=0.4)
    sns.histplot(errors[y_obf == 1], color="red", label="Offuscato", kde=True, stat="density", alpha=0.4)
    plt.title("Distribuzione Reconstruction Error — VAE"); plt.legend()
    _save_fig(os.path.join(output_dir, "vae_error_distribution.png"))

    df_err = pd.DataFrame({"Errore": errors, "Classe": y_obf}).replace({"Classe": {0: "Non offuscato", 1: "Offuscato"}})
    plt.figure(figsize=(7, 5))
    sns.boxplot(x="Classe", y="Errore", data=df_err, hue="Classe",
                palette={"Non offuscato": "blue", "Offuscato": "red"}, legend=False)
    plt.title("Reconstruction Error per Classe")
    _save_fig(os.path.join(output_dir, "vae_error_boxplot.png"))

    fpr, tpr, _ = roc_curve(y_obf, errors)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC={auc(fpr, tpr):.4f}"); plt.plot([0, 1], [0, 1], "k--")
    plt.title("ROC — VAE"); plt.legend()
    _save_fig(os.path.join(output_dir, "vae_roc_curve.png"))

    prec, rec, _ = precision_recall_curve(y_obf, errors)
    plt.figure(figsize=(6, 5))
    plt.plot(rec, prec); plt.title("PR Curve — VAE")
    _save_fig(os.path.join(output_dir, "vae_pr_curve.png"))

    plt.figure(figsize=(7, 5))
    plt.scatter(errors, pct, c=y_obf, cmap="coolwarm", alpha=0.5, s=10)
    plt.title("Reconstruction Error vs Percentuale Calibrata"); plt.colorbar(label="is_obfuscated")
    _save_fig(os.path.join(output_dir, "vae_error_vs_pct.png"))


def main_train_vae():
    t_start = time.time()
    csv_scaler = joblib.load(os.path.join(CNN_MODEL_DIR, "csv_scaler.pkl"))
    obf_labeler = joblib.load(os.path.join(CNN_MODEL_DIR, "obf_labeler.pkl"))

    train_df = apply_obfuscation_labels(pd.read_csv(os.path.join(BASE_DIR, "dataset_train.csv")), obf_labeler)
    val_df = apply_obfuscation_labels(pd.read_csv(os.path.join(BASE_DIR, "dataset_val.csv")), obf_labeler)
    obf_train = train_df.set_index("filename")["is_obfuscated"].to_dict()
    obf_val = val_df.set_index("filename")["is_obfuscated"].to_dict()

    train_ds = make_dataset("train", obf_train, csv_scaler, shuffle=True, with_sample_weight=False)
    val_ds = make_dataset("val", obf_val, csv_scaler, shuffle=False, with_sample_weight=False)

    best_model_path = os.path.join(CNN_MODEL_DIR, "best_model.keras")
    if not os.path.isfile(best_model_path):
        raise FileNotFoundError(f"CNN non trovata in {best_model_path}. Esegui prima la cella CNN.")
    cnn_model = load_trained_cnn(best_model_path)

    train_emb, _, _ = extract_fusion_embeddings(cnn_model, train_ds)
    val_emb, _, y_obf_val = extract_fusion_embeddings(cnn_model, val_ds)

    vae, train_losses, val_losses = train_vae_on_embeddings(train_emb, val_emb)

    vae.encoder.save(os.path.join(VAE_MODEL_DIR, "vae_encoder.keras"))
    vae.decoder.save(os.path.join(VAE_MODEL_DIR, "vae_decoder.keras"))

    train_errors = compute_vae_reconstruction_errors(vae, train_emb)
    err_min, err_max = float(train_errors.min()), float(train_errors.max())
    with open(os.path.join(VAE_MODEL_DIR, "vae_calibration.json"), "w") as f:
        json.dump({"err_min": err_min, "err_max": err_max, "embedding_source": "fusion_bn",
                    "latent_dim": VAE_LATENT_DIM}, f)

    val_errors = compute_vae_reconstruction_errors(vae, val_emb)
    val_pct = errors_to_percentage(val_errors, err_min, err_max)
    print(f"VAE ROC-AUC (val): {roc_auc_score(y_obf_val, val_errors):.4f}")
    plot_vae_results(train_losses, val_losses, val_errors, y_obf_val, val_pct, VAE_TRAIN_DIR)
    print(f"Training VAE completato in {(time.time() - t_start) / 60:.1f} min")
    return vae


def main_test_vae():
    t_start = time.time()
    csv_scaler, obf_labeler = load_scaler_and_labeler()
    test_ds_eval = load_test_dataset(csv_scaler, obf_labeler)
    cnn_model, _ = evaluate_cnn_on_test(test_ds_eval)

    encoder_path = os.path.join(VAE_MODEL_DIR, "vae_encoder.keras")
    decoder_path = os.path.join(VAE_MODEL_DIR, "vae_decoder.keras")
    calib_path = os.path.join(VAE_MODEL_DIR, "vae_calibration.json")
    if not all(os.path.isfile(p) for p in [encoder_path, decoder_path, calib_path]):
        raise FileNotFoundError("VAE non trovato. Esegui prima main_train_vae().")

    encoder = tf.keras.models.load_model(encoder_path, compile=False, custom_objects={"Sampling": Sampling})
    decoder = tf.keras.models.load_model(decoder_path, compile=False)
    vae_model = VAE(encoder, decoder)
    with open(calib_path) as f:
        calib = json.load(f)

    test_emb, _, y_obf_test = extract_fusion_embeddings(cnn_model, test_ds_eval)
    test_errors = compute_vae_reconstruction_errors(vae_model, test_emb)
    test_pct = errors_to_percentage(test_errors, calib["err_min"], calib["err_max"])

    print(f"VAE ROC-AUC (test): {roc_auc_score(y_obf_test, test_errors):.4f}")
    plot_vae_results([], [], test_errors, y_obf_test, test_pct, VAE_TEST_DIR)
    csv_out = save_unsupervised_predictions(test_pct, VAE_TEST_DIR, "obfuscation_percentage_test_vae.csv")
    print(f"Test VAE completato in {(time.time() - t_start) / 60:.1f} min | CSV: {csv_out}")


# --- Funzioni minime dalla cella CNN, necessarie a VAE ---

EFFICIENTNET_WEIGHTS_PATH = os.path.join(CNN_MODEL_DIR, "efficientnetb0_notop.h5")


def build_csv_branch(csv_in_dim, dropout=0.30):
    inp = layers.Input(shape=(csv_in_dim,), name="csv_feat_input")
    x = layers.Dense(256, activation="swish")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    residual = x
    x = layers.Dense(256, activation="swish")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Add()([x, residual])
    x = layers.Dense(128, activation="swish")(x)
    x = layers.BatchNormalization()(x)
    return models.Model(inp, x, name="CSVResidualMLP")


def build_malware_cnn(csv_in_dim):
    image_in = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name="image")
    csv_in = layers.Input(shape=(csv_in_dim,), name="csv_feat")

    x_img = layers.RandomTranslation(0.05, 0.05, fill_mode="constant", fill_value=0.0)(image_in)
    x_img = layers.RandomContrast(0.10)(x_img)
    x_img = layers.GaussianNoise(0.02)(x_img)
    x_img = layers.Concatenate(name="gray_to_rgb")([x_img, x_img, x_img])

    backbone = tf.keras.applications.EfficientNetB0(
        include_top=False, weights=None,
        input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg", name="EfficientNetB0"
    )
    if os.path.isfile(EFFICIENTNET_WEIGHTS_PATH):
        backbone.load_weights(EFFICIENTNET_WEIGHTS_PATH)
        print(f"Pesi ImageNet caricati da {EFFICIENTNET_WEIGHTS_PATH}")
    else:
        print(f"ATTENZIONE: {EFFICIENTNET_WEIGHTS_PATH} non trovato — backbone non pre-addestrato (pesi random).")

    backbone.trainable = False
    x_img = backbone(x_img)

    x_img = layers.BatchNormalization(name="image_projection_bn")(x_img)
    x_img = layers.Dense(256, activation="swish", name="image_projection")(x_img)
    x_img = layers.Dropout(DROPOUT)(x_img)

    x_csv = build_csv_branch(csv_in_dim, dropout=DROPOUT)(csv_in)
    x_csv = layers.Dense(256, activation="swish", name="csv_projection")(x_csv)
    x_csv = layers.BatchNormalization(name="csv_projection_bn")(x_csv)

    gate = layers.Dense(256, activation="sigmoid", name="feature_gate")(layers.Concatenate()([x_img, x_csv]))
    x_img_gated = layers.Multiply()([x_img, gate])
    x_csv_gated = layers.Multiply()([x_csv, 1.0 - gate])

    shared = layers.Concatenate(name="multimodal_fusion")([x_img_gated, x_csv_gated])
    shared = layers.Dense(256, activation="swish", name="fusion_dense")(shared)
    shared = layers.BatchNormalization(name="fusion_bn")(shared)
    shared = layers.Dropout(DROPOUT)(shared)

    mal = layers.Dense(128, activation="swish")(shared)
    mal = layers.BatchNormalization()(mal)
    mal = layers.Dropout(0.20)(mal)
    out_mal = layers.Dense(1, activation="sigmoid", dtype="float32", name="malware")(mal)

    obf = layers.Dense(128, activation="swish")(shared)
    obf = layers.BatchNormalization()(obf)
    obf = layers.Dropout(0.20)(obf)
    out_obf = layers.Dense(1, activation="sigmoid", dtype="float32", name="obfuscated")(obf)

    model = models.Model(inputs=[image_in, csv_in],
                          outputs={"malware": out_mal, "obfuscated": out_obf},
                          name="MalwareEfficientNetB0")
    return model, backbone


def load_trained_cnn(model_path):
    import zipfile
    cnn_model, _ = build_malware_cnn(CSV_DIM)
    with zipfile.ZipFile(model_path, "r") as z:
        weights_name = next((n for n in z.namelist() if n.endswith(".weights.h5")), None)
        if weights_name is None:
            raise FileNotFoundError(f"Nessun .weights.h5 in {model_path}")
        extract_dir = "/tmp/cnn_extract"
        z.extract(weights_name, extract_dir)
    cnn_model.load_weights(os.path.join(extract_dir, weights_name))
    return cnn_model


def extract_fusion_embeddings(cnn_model, dataset):
    embedder = models.Model(cnn_model.input, cnn_model.get_layer("fusion_bn").output)
    embs, y_mal_list, y_obf_list = [], [], []
    for inputs, targets in dataset:
        embs.append(tf.cast(embedder(inputs, training=False), tf.float32).numpy())
        y_mal_list.append(targets["malware"].numpy())
        y_obf_list.append(targets["obfuscated"].numpy())
    return np.concatenate(embs), np.concatenate(y_mal_list), np.concatenate(y_obf_list)


def get_predictions(model, dataset):
    labels_mal, labels_obf, probs_mal, probs_obf = [], [], [], []
    for inputs, targets in dataset:
        preds = model(inputs, training=False)
        labels_mal.append(targets["malware"].numpy())
        labels_obf.append(targets["obfuscated"].numpy())
        probs_mal.append(np.asarray(preds["malware"]).flatten())
        probs_obf.append(np.asarray(preds["obfuscated"]).flatten())
    return {
        "labels_mal": np.concatenate(labels_mal).flatten(),
        "probs_mal": np.concatenate(probs_mal),
        "labels_obf": np.concatenate(labels_obf).flatten(),
        "probs_obf": np.concatenate(probs_obf)
    }


def print_classification_report(res):
    print("\n--- MALWARE ---")
    print(classification_report(res["labels_mal"], (res["probs_mal"] > 0.5).astype(int),
                                 target_names=["Goodware", "Malware"]))
    print(f"ROC-AUC: {roc_auc_score(res['labels_mal'], res['probs_mal']):.4f}")
    print("\n--- OFFUSCAMENTO ---")
    print(classification_report(res["labels_obf"], (res["probs_obf"] > 0.5).astype(int),
                                 target_names=["Non offuscato", "Offuscato"]))
    print(f"ROC-AUC: {roc_auc_score(res['labels_obf'], res['probs_obf']):.4f}")


def plot_confusion_matrix(labels, preds, title, out_path, cmap="Blues"):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap)
    plt.title(title); plt.xlabel("Predetto"); plt.ylabel("Reale")
    _save_fig(out_path)


def save_plots(history, res, output_dir):
    pm, lm = np.array(res["probs_mal"]), np.array(res["labels_mal"])
    po, lo = np.array(res["probs_obf"]), np.array(res["labels_obf"])

    if history is not None and "loss" in history.history:
        plt.figure(figsize=(8, 4))
        plt.plot(history.history["loss"], label="Train")
        plt.plot(history.history["val_loss"], label="Val")
        plt.title("Loss per Epoca"); plt.xlabel("Epoca"); plt.ylabel("Loss"); plt.legend()
        _save_fig(os.path.join(output_dir, "loss_curve.png"))

    plot_confusion_matrix(lm, (pm > 0.5).astype(int), "Confusion Matrix - Malware",
                           os.path.join(output_dir, "confusion_matrix_malware.png"))
    plot_confusion_matrix(lo, (po > 0.5).astype(int), "Confusion Matrix - Offuscato",
                           os.path.join(output_dir, "confusion_matrix_obfuscation.png"), cmap="Oranges")

    df_res = pd.DataFrame({"Confidenza": pm, "Classe": lm}).replace({"Classe": {0: "Goodware", 1: "Malware"}})
    plt.figure(figsize=(8, 6))
    sns.boxplot(x="Classe", y="Confidenza", data=df_res, hue="Classe",
                palette={"Goodware": "blue", "Malware": "red"}, legend=False)
    plt.axhline(0.5, color="black", linestyle="--")
    _save_fig(os.path.join(output_dir, "confidence_boxplot_malware.png"))

    for name, labels, probs, fname in [("Malware", lm, pm, "malware"), ("Offuscamento", lo, po, "obfuscation")]:
        fpr, tpr, _ = roc_curve(labels, probs)
        plt.figure(figsize=(6, 5))
        plt.plot(fpr, tpr, label=f"AUC={auc(fpr, tpr):.4f}")
        plt.plot([0, 1], [0, 1], "k--")
        plt.title(f"ROC - {name}"); plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.legend()
        _save_fig(os.path.join(output_dir, f"roc_curve_{fname}.png"))

        prec, rec, _ = precision_recall_curve(labels, probs)
        plt.figure(figsize=(6, 5))
        plt.plot(rec, prec)
        plt.title(f"PR Curve - {name}"); plt.xlabel("Recall"); plt.ylabel("Precision")
        _save_fig(os.path.join(output_dir, f"pr_curve_{fname}.png"))


def evaluate_cnn_on_test(test_ds_no_weight):
    cnn_model = load_trained_cnn(os.path.join(CNN_MODEL_DIR, "best_model.keras"))
    res = get_predictions(cnn_model, test_ds_no_weight)
    print_classification_report(res)
    save_plots(None, res, output_dir=CNN_TEST_DIR)
    return cnn_model, res

# --- Fine funzioni riportate dalla cella CNN ---


vae_model = main_train_vae()
main_test_vae()

Pesi ImageNet caricati da /content/drive/MyDrive/ProgettoSicurezzaReti/cnn_model/model/efficientnetb0_notop.h5
Epoch 1/100
563/563 - 11s - 20ms/step - kl_loss: 119.3089 - loss: 91.7170 - recon_loss: 91.5977 - val_kl_loss: 1251.3165 - val_loss: 5202.2788 - val_recon_loss: 5201.0283
Epoch 2/100
563/563 - 3s - 5ms/step - kl_loss: 176.5742 - loss: 55.9396 - recon_loss: 55.7630 - val_kl_loss: 201.0681 - val_loss: 69.3632 - val_recon_loss: 69.1621
Epoch 3/100
563/563 - 2s - 4ms/step - kl_loss: 193.8227 - loss: 48.2891 - recon_loss: 48.0952 - val_kl_loss: 216.8270 - val_loss: 46.8411 - val_recon_loss: 46.6243
Epoch 4/100
563/563 - 2s - 4ms/step - kl_loss: 201.8377 - loss: 42.8547 - recon_loss: 42.6528 - val_kl_loss: 211.4705 - val_loss: 46.2668 - val_recon_loss: 46.0553
Epoch 5/100
563/563 - 2s - 4ms/step - kl_loss: 203.8447 - loss: 41.0365 - recon_loss: 40.8326 - val_kl_loss: 210.3579 - val_loss: 42.7214 - val_recon_loss: 42.5110
Epoch 6/100
563/563 - 2s - 3ms/step - kl_loss: 204.9829 - loss

/tmp/ipykernel_2833/269505058.py:78: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout(); plt.savefig(path); plt.close()


Training VAE completato in 14.0 min
Pesi ImageNet caricati da /content/drive/MyDrive/ProgettoSicurezzaReti/cnn_model/model/efficientnetb0_notop.h5

--- MALWARE ---
              precision    recall  f1-score   support

    Goodware       1.00      1.00      1.00      5684
     Malware       1.00      1.00      1.00      2028

    accuracy                           1.00      7712
   macro avg       1.00      1.00      1.00      7712
weighted avg       1.00      1.00      1.00      7712

ROC-AUC: 1.0000

--- OFFUSCAMENTO ---
               precision    recall  f1-score   support

Non offuscato       1.00      0.99      1.00      6961
    Offuscato       0.95      0.97      0.96       751

     accuracy                           0.99      7712
    macro avg       0.98      0.98      0.98      7712
 weighted avg       0.99      0.99      0.99      7712

ROC-AUC: 0.9976
VAE ROC-AUC (test): 0.5626


/tmp/ipykernel_2833/269505058.py:78: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout(); plt.savefig(path); plt.close()


Test VAE completato in 4.2 min | CSV: /content/drive/MyDrive/ProgettoSicurezzaReti/vae_model/test/obfuscation_percentage_test_vae.csv


In [ ]:
# ==============================================================
# Dipendenze CNN (ridefinite qui solo se non già presenti in memoria,
# così la cella funziona anche se non hai eseguito prima la cella CNN
# in questa sessione — es. dopo un restart del runtime).
# ==============================================================
EFFICIENTNET_WEIGHTS_PATH = os.path.join(CNN_MODEL_DIR, "efficientnetb0_notop.h5")

if "build_csv_branch" not in globals():
    def build_csv_branch(csv_in_dim, dropout=0.30):
        inp = layers.Input(shape=(csv_in_dim,), name="csv_feat_input")
        x = layers.Dense(256, activation="swish")(inp)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout)(x)
        residual = x
        x = layers.Dense(256, activation="swish")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout)(x)
        x = layers.Add()([x, residual])
        x = layers.Dense(128, activation="swish")(x)
        x = layers.BatchNormalization()(x)
        return models.Model(inp, x, name="CSVResidualMLP")


if "build_malware_cnn" not in globals():
    def build_malware_cnn(csv_in_dim):
        image_in = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name="image")
        csv_in = layers.Input(shape=(csv_in_dim,), name="csv_feat")

        x_img = layers.RandomTranslation(0.05, 0.05, fill_mode="constant", fill_value=0.0)(image_in)
        x_img = layers.RandomContrast(0.10)(x_img)
        x_img = layers.GaussianNoise(0.02)(x_img)
        x_img = layers.Concatenate(name="gray_to_rgb")([x_img, x_img, x_img])

        backbone = tf.keras.applications.EfficientNetB0(
            include_top=False, weights=None,
            input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg", name="EfficientNetB0"
        )
        if os.path.isfile(EFFICIENTNET_WEIGHTS_PATH):
            backbone.load_weights(EFFICIENTNET_WEIGHTS_PATH)
            print(f"Pesi ImageNet caricati da {EFFICIENTNET_WEIGHTS_PATH}")
        else:
            print(f"ATTENZIONE: {EFFICIENTNET_WEIGHTS_PATH} non trovato — backbone non pre-addestrato (pesi random).")

        backbone.trainable = False
        x_img = backbone(x_img)

        x_img = layers.BatchNormalization(name="image_projection_bn")(x_img)
        x_img = layers.Dense(256, activation="swish", name="image_projection")(x_img)
        x_img = layers.Dropout(DROPOUT)(x_img)

        x_csv = build_csv_branch(csv_in_dim, dropout=DROPOUT)(csv_in)
        x_csv = layers.Dense(256, activation="swish", name="csv_projection")(x_csv)
        x_csv = layers.BatchNormalization(name="csv_projection_bn")(x_csv)

        gate = layers.Dense(256, activation="sigmoid", name="feature_gate")(layers.Concatenate()([x_img, x_csv]))
        x_img_gated = layers.Multiply()([x_img, gate])
        x_csv_gated = layers.Multiply()([x_csv, 1.0 - gate])

        shared = layers.Concatenate(name="multimodal_fusion")([x_img_gated, x_csv_gated])
        shared = layers.Dense(256, activation="swish", name="fusion_dense")(shared)
        shared = layers.BatchNormalization(name="fusion_bn")(shared)
        shared = layers.Dropout(DROPOUT)(shared)

        mal = layers.Dense(128, activation="swish")(shared)
        mal = layers.BatchNormalization()(mal)
        mal = layers.Dropout(0.20)(mal)
        out_mal = layers.Dense(1, activation="sigmoid", dtype="float32", name="malware")(mal)

        obf = layers.Dense(128, activation="swish")(shared)
        obf = layers.BatchNormalization()(obf)
        obf = layers.Dropout(0.20)(obf)
        out_obf = layers.Dense(1, activation="sigmoid", dtype="float32", name="obfuscated")(obf)

        model = models.Model(inputs=[image_in, csv_in],
                              outputs={"malware": out_mal, "obfuscated": out_obf},
                              name="MalwareEfficientNetB0")
        return model, backbone


if "load_trained_cnn" not in globals():
    def load_trained_cnn(model_path):
        import zipfile
        cnn_model, _ = build_malware_cnn(CSV_DIM)
        with zipfile.ZipFile(model_path, "r") as z:
            weights_name = next((n for n in z.namelist() if n.endswith(".weights.h5")), None)
            if weights_name is None:
                raise FileNotFoundError(f"Nessun .weights.h5 in {model_path}")
            extract_dir = "/tmp/cnn_extract"
            z.extract(weights_name, extract_dir)
        cnn_model.load_weights(os.path.join(extract_dir, weights_name))
        return cnn_model


if "extract_fusion_embeddings" not in globals():
    def extract_fusion_embeddings(cnn_model, dataset):
        embedder = models.Model(cnn_model.input, cnn_model.get_layer("fusion_bn").output)
        embs, y_mal_list, y_obf_list = [], [], []
        for inputs, targets in dataset:
            embs.append(tf.cast(embedder(inputs, training=False), tf.float32).numpy())
            y_mal_list.append(targets["malware"].numpy())
            y_obf_list.append(targets["obfuscated"].numpy())
        return np.concatenate(embs), np.concatenate(y_mal_list), np.concatenate(y_obf_list)


if "get_predictions" not in globals():
    def get_predictions(model, dataset):
        labels_mal, labels_obf, probs_mal, probs_obf = [], [], [], []
        for inputs, targets in dataset:
            preds = model(inputs, training=False)
            labels_mal.append(targets["malware"].numpy())
            labels_obf.append(targets["obfuscated"].numpy())
            probs_mal.append(np.asarray(preds["malware"]).flatten())
            probs_obf.append(np.asarray(preds["obfuscated"]).flatten())
        return {
            "labels_mal": np.concatenate(labels_mal).flatten(),
            "probs_mal": np.concatenate(probs_mal),
            "labels_obf": np.concatenate(labels_obf).flatten(),
            "probs_obf": np.concatenate(probs_obf)
        }


if "print_classification_report" not in globals():
    def print_classification_report(res):
        print("\n--- MALWARE ---")
        print(classification_report(res["labels_mal"], (res["probs_mal"] > 0.5).astype(int),
                                     target_names=["Goodware", "Malware"]))
        print(f"ROC-AUC: {roc_auc_score(res['labels_mal'], res['probs_mal']):.4f}")
        print("\n--- OFFUSCAMENTO ---")
        print(classification_report(res["labels_obf"], (res["probs_obf"] > 0.5).astype(int),
                                     target_names=["Non offuscato", "Offuscato"]))
        print(f"ROC-AUC: {roc_auc_score(res['labels_obf'], res['probs_obf']):.4f}")


if "plot_confusion_matrix" not in globals():
    def plot_confusion_matrix(labels, preds, title, out_path, cmap="Blues"):
        cm = confusion_matrix(labels, preds)
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt="d", cmap=cmap)
        plt.title(title); plt.xlabel("Predetto"); plt.ylabel("Reale")
        _save_fig(out_path)


if "save_plots" not in globals():
    def save_plots(history, res, output_dir):
        pm, lm = np.array(res["probs_mal"]), np.array(res["labels_mal"])
        po, lo = np.array(res["probs_obf"]), np.array(res["labels_obf"])

        if history is not None and "loss" in history.history:
            plt.figure(figsize=(8, 4))
            plt.plot(history.history["loss"], label="Train")
            plt.plot(history.history["val_loss"], label="Val")
            plt.title("Loss per Epoca"); plt.xlabel("Epoca"); plt.ylabel("Loss"); plt.legend()
            _save_fig(os.path.join(output_dir, "loss_curve.png"))

        plot_confusion_matrix(lm, (pm > 0.5).astype(int), "Confusion Matrix - Malware",
                               os.path.join(output_dir, "confusion_matrix_malware.png"))
        plot_confusion_matrix(lo, (po > 0.5).astype(int), "Confusion Matrix - Offuscato",
                               os.path.join(output_dir, "confusion_matrix_obfuscation.png"), cmap="Oranges")

        df_res = pd.DataFrame({"Confidenza": pm, "Classe": lm}).replace({"Classe": {0: "Goodware", 1: "Malware"}})
        plt.figure(figsize=(8, 6))
        sns.boxplot(x="Classe", y="Confidenza", data=df_res, hue="Classe",
                    palette={"Goodware": "blue", "Malware": "red"}, legend=False)
        plt.axhline(0.5, color="black", linestyle="--")
        _save_fig(os.path.join(output_dir, "confidence_boxplot_malware.png"))

        for name, labels, probs, fname in [("Malware", lm, pm, "malware"), ("Offuscamento", lo, po, "obfuscation")]:
            fpr, tpr, _ = roc_curve(labels, probs)
            plt.figure(figsize=(6, 5))
            plt.plot(fpr, tpr, label=f"AUC={auc(fpr, tpr):.4f}")
            plt.plot([0, 1], [0, 1], "k--")
            plt.title(f"ROC - {name}"); plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.legend()
            _save_fig(os.path.join(output_dir, f"roc_curve_{fname}.png"))

            prec, rec, _ = precision_recall_curve(labels, probs)
            plt.figure(figsize=(6, 5))
            plt.plot(rec, prec)
            plt.title(f"PR Curve - {name}"); plt.xlabel("Recall"); plt.ylabel("Precision")
            _save_fig(os.path.join(output_dir, f"pr_curve_{fname}.png"))


if "evaluate_cnn_on_test" not in globals():
    def evaluate_cnn_on_test(test_ds_no_weight):
        cnn_model = load_trained_cnn(os.path.join(CNN_MODEL_DIR, "best_model.keras"))
        res = get_predictions(cnn_model, test_ds_no_weight)
        print_classification_report(res)
        save_plots(None, res, output_dir=CNN_TEST_DIR)
        return cnn_model, res


# ==============================================================
# LOCAL OUTLIER FACTOR
# ==============================================================

LOF_DIR = os.path.join(BASE_DIR, "lof_model")
LOF_MODEL_DIR, LOF_TRAIN_DIR, LOF_TEST_DIR = (os.path.join(LOF_DIR, s) for s in ["model", "train", "test"])
for d in [LOF_MODEL_DIR, LOF_TRAIN_DIR, LOF_TEST_DIR]:
    os.makedirs(d, exist_ok=True)

from sklearn.neighbors import LocalOutlierFactor

LOF_CONTAMINATION = 0.15
LOF_N_NEIGHBORS_GRID = [10, 20, 30, 50]


def plot_lof_results(scores, y_obf, pct, output_dir):
    plt.figure(figsize=(8, 5))
    sns.histplot(scores[y_obf == 0], color="blue", label="Non offuscato", kde=True, stat="density", alpha=0.4)
    sns.histplot(scores[y_obf == 1], color="red", label="Offuscato", kde=True, stat="density", alpha=0.4)
    plt.title("Distribuzione Anomaly Score — LOF"); plt.legend()
    _save_fig(os.path.join(output_dir, "lof_score_distribution.png"))

    df_score = pd.DataFrame({"Score": scores, "Classe": y_obf}).replace({"Classe": {0: "Non offuscato", 1: "Offuscato"}})
    plt.figure(figsize=(7, 5))
    sns.boxplot(x="Classe", y="Score", data=df_score, hue="Classe",
                palette={"Non offuscato": "blue", "Offuscato": "red"}, legend=False)
    plt.title("Anomaly Score per Classe — LOF")
    _save_fig(os.path.join(output_dir, "lof_score_boxplot.png"))

    fpr, tpr, _ = roc_curve(y_obf, scores)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC={auc(fpr, tpr):.4f}"); plt.plot([0, 1], [0, 1], "k--")
    plt.title("ROC — LOF"); plt.legend()
    _save_fig(os.path.join(output_dir, "lof_roc_curve.png"))

    prec, rec, _ = precision_recall_curve(y_obf, scores)
    plt.figure(figsize=(6, 5))
    plt.plot(rec, prec); plt.title("PR Curve — LOF")
    _save_fig(os.path.join(output_dir, "lof_pr_curve.png"))

    plt.figure(figsize=(7, 5))
    plt.scatter(scores, pct, c=y_obf, cmap="coolwarm", alpha=0.5, s=10)
    plt.title("Score vs Percentuale Calibrata — LOF"); plt.colorbar(label="is_obfuscated")
    _save_fig(os.path.join(output_dir, "lof_score_vs_pct.png"))


def select_lof_n_neighbors(train_emb, y_obf_train, grid=LOF_N_NEIGHBORS_GRID):
    best_k, best_auc = grid[0], -1.0
    for k in grid:
        lof = LocalOutlierFactor(n_neighbors=k, contamination=LOF_CONTAMINATION, novelty=True, n_jobs=-1)
        lof.fit(train_emb)
        scores = -lof.score_samples(train_emb)
        try:
            score_auc = roc_auc_score(y_obf_train, scores)
        except ValueError:
            score_auc = -1.0
        print(f"  n_neighbors={k}: AUC(train)={score_auc:.4f}")
        if score_auc > best_auc:
            best_auc, best_k = score_auc, k
    print(f"Selezionato n_neighbors={best_k} (AUC train={best_auc:.4f})")
    return best_k


def train_embedding_lof(train_emb, y_obf_train):
    t0 = time.time()
    best_k = select_lof_n_neighbors(train_emb, y_obf_train)
    lof = LocalOutlierFactor(n_neighbors=best_k, contamination=LOF_CONTAMINATION, novelty=True, n_jobs=-1)
    lof.fit(train_emb)
    print(f"LOF fit completato in {time.time() - t0:.1f}s")
    return lof, best_k


def compute_lof_anomaly_scores(lof, embeddings):
    return -lof.score_samples(embeddings)


def main_train_lof():
    t_start = time.time()
    _, csv_scaler, obf_dicts = fit_and_save_scalers()
    train_ds = make_dataset("train", obf_dicts["train"], csv_scaler, shuffle=True, with_sample_weight=False)

    best_model_path = os.path.join(CNN_MODEL_DIR, "best_model.keras")
    if not os.path.isfile(best_model_path):
        raise FileNotFoundError(f"CNN non trovata in {best_model_path}. Esegui prima la cella CNN.")
    cnn_model = load_trained_cnn(best_model_path)

    train_emb, _, y_obf_train = extract_fusion_embeddings(cnn_model, train_ds)
    lof_model, best_k = train_embedding_lof(train_emb, y_obf_train)
    joblib.dump(lof_model, os.path.join(LOF_MODEL_DIR, "lof_embedding.pkl"))

    train_scores = compute_lof_anomaly_scores(lof_model, train_emb)
    score_min, score_max = float(train_scores.min()), float(train_scores.max())
    with open(os.path.join(LOF_MODEL_DIR, "lof_calibration.json"), "w") as f:
        json.dump({"score_min": score_min, "score_max": score_max, "embedding_source": "fusion_bn",
                    "n_neighbors": best_k, "contamination": LOF_CONTAMINATION}, f)

    train_pct = errors_to_percentage(train_scores, score_min, score_max)
    plot_lof_results(train_scores, y_obf_train, train_pct, LOF_TRAIN_DIR)
    print(f"LOF ROC-AUC (train): {roc_auc_score(y_obf_train, train_scores):.4f}")
    print(f"Training LOF completato in {(time.time() - t_start) / 60:.1f} min")
    return lof_model


def main_test_lof():
    t_start = time.time()
    csv_scaler, obf_labeler = load_scaler_and_labeler()
    test_ds_eval = load_test_dataset(csv_scaler, obf_labeler)
    cnn_model, _ = evaluate_cnn_on_test(test_ds_eval)

    lof_model = joblib.load(os.path.join(LOF_MODEL_DIR, "lof_embedding.pkl"))
    with open(os.path.join(LOF_MODEL_DIR, "lof_calibration.json")) as f:
        calib = json.load(f)

    test_emb, _, y_obf_test = extract_fusion_embeddings(cnn_model, test_ds_eval)
    test_scores = compute_lof_anomaly_scores(lof_model, test_emb)
    test_pct = errors_to_percentage(test_scores, calib["score_min"], calib["score_max"])

    print(f"LOF ROC-AUC (test): {roc_auc_score(y_obf_test, test_scores):.4f}")
    plot_lof_results(test_scores, y_obf_test, test_pct, LOF_TEST_DIR)
    csv_out = save_unsupervised_predictions(test_pct, LOF_TEST_DIR, "obfuscation_percentage_test_lof.csv")
    print(f"Test LOF completato in {(time.time() - t_start) / 60:.1f} min | CSV: {csv_out}")


lof_model = main_train_lof()
main_test_lof()

Train: 36047 | Val: 7725 | Test: 7723
Offuscati train: 3571/36047 (9.9%)
Pesi ImageNet caricati da /content/drive/MyDrive/ProgettoSicurezzaReti/cnn_model/model/efficientnetb0_notop.h5
  n_neighbors=10: AUC(train)=0.5328
  n_neighbors=20: AUC(train)=0.5617
  n_neighbors=30: AUC(train)=0.5948
  n_neighbors=50: AUC(train)=0.6008
Selezionato n_neighbors=50 (AUC train=0.6008)
LOF fit completato in 216.6s
LOF ROC-AUC (train): 0.6008
Training LOF completato in 13.5 min
Pesi ImageNet caricati da /content/drive/MyDrive/ProgettoSicurezzaReti/cnn_model/model/efficientnetb0_notop.h5

--- MALWARE ---
              precision    recall  f1-score   support

    Goodware       1.00      1.00      1.00      5684
     Malware       1.00      1.00      1.00      2028

    accuracy                           1.00      7712
   macro avg       1.00      1.00      1.00      7712
weighted avg       1.00      1.00      1.00      7712

ROC-AUC: 1.0000

--- OFFUSCAMENTO ---
               precision    recall  f1-s

/tmp/ipykernel_1914/269505058.py:78: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout(); plt.savefig(path); plt.close()


Test LOF completato in 5.2 min | CSV: /content/drive/MyDrive/ProgettoSicurezzaReti/lof_model/test/obfuscation_percentage_test_lof.csv


In [ ]:
# ==============================================================
# Dipendenze CNN (ridefinite qui solo se non già presenti in memoria,
# così la cella funziona anche se non hai eseguito prima la cella CNN
# in questa sessione — es. dopo un restart del runtime).
# ==============================================================
EFFICIENTNET_WEIGHTS_PATH = os.path.join(CNN_MODEL_DIR, "efficientnetb0_notop.h5")

if "build_csv_branch" not in globals():
    def build_csv_branch(csv_in_dim, dropout=0.30):
        inp = layers.Input(shape=(csv_in_dim,), name="csv_feat_input")
        x = layers.Dense(256, activation="swish")(inp)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout)(x)
        residual = x
        x = layers.Dense(256, activation="swish")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout)(x)
        x = layers.Add()([x, residual])
        x = layers.Dense(128, activation="swish")(x)
        x = layers.BatchNormalization()(x)
        return models.Model(inp, x, name="CSVResidualMLP")


if "build_malware_cnn" not in globals():
    def build_malware_cnn(csv_in_dim):
        image_in = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name="image")
        csv_in = layers.Input(shape=(csv_in_dim,), name="csv_feat")

        x_img = layers.RandomTranslation(0.05, 0.05, fill_mode="constant", fill_value=0.0)(image_in)
        x_img = layers.RandomContrast(0.10)(x_img)
        x_img = layers.GaussianNoise(0.02)(x_img)
        x_img = layers.Concatenate(name="gray_to_rgb")([x_img, x_img, x_img])

        backbone = tf.keras.applications.EfficientNetB0(
            include_top=False, weights=None,
            input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg", name="EfficientNetB0"
        )
        if os.path.isfile(EFFICIENTNET_WEIGHTS_PATH):
            backbone.load_weights(EFFICIENTNET_WEIGHTS_PATH)
            print(f"Pesi ImageNet caricati da {EFFICIENTNET_WEIGHTS_PATH}")
        else:
            print(f"ATTENZIONE: {EFFICIENTNET_WEIGHTS_PATH} non trovato — backbone non pre-addestrato (pesi random).")

        backbone.trainable = False
        x_img = backbone(x_img)

        x_img = layers.BatchNormalization(name="image_projection_bn")(x_img)
        x_img = layers.Dense(256, activation="swish", name="image_projection")(x_img)
        x_img = layers.Dropout(DROPOUT)(x_img)

        x_csv = build_csv_branch(csv_in_dim, dropout=DROPOUT)(csv_in)
        x_csv = layers.Dense(256, activation="swish", name="csv_projection")(x_csv)
        x_csv = layers.BatchNormalization(name="csv_projection_bn")(x_csv)

        gate = layers.Dense(256, activation="sigmoid", name="feature_gate")(layers.Concatenate()([x_img, x_csv]))
        x_img_gated = layers.Multiply()([x_img, gate])
        x_csv_gated = layers.Multiply()([x_csv, 1.0 - gate])

        shared = layers.Concatenate(name="multimodal_fusion")([x_img_gated, x_csv_gated])
        shared = layers.Dense(256, activation="swish", name="fusion_dense")(shared)
        shared = layers.BatchNormalization(name="fusion_bn")(shared)
        shared = layers.Dropout(DROPOUT)(shared)

        mal = layers.Dense(128, activation="swish")(shared)
        mal = layers.BatchNormalization()(mal)
        mal = layers.Dropout(0.20)(mal)
        out_mal = layers.Dense(1, activation="sigmoid", dtype="float32", name="malware")(mal)

        obf = layers.Dense(128, activation="swish")(shared)
        obf = layers.BatchNormalization()(obf)
        obf = layers.Dropout(0.20)(obf)
        out_obf = layers.Dense(1, activation="sigmoid", dtype="float32", name="obfuscated")(obf)

        model = models.Model(inputs=[image_in, csv_in],
                              outputs={"malware": out_mal, "obfuscated": out_obf},
                              name="MalwareEfficientNetB0")
        return model, backbone


if "load_trained_cnn" not in globals():
    def load_trained_cnn(model_path):
        import zipfile
        cnn_model, _ = build_malware_cnn(CSV_DIM)
        with zipfile.ZipFile(model_path, "r") as z:
            weights_name = next((n for n in z.namelist() if n.endswith(".weights.h5")), None)
            if weights_name is None:
                raise FileNotFoundError(f"Nessun .weights.h5 in {model_path}")
            extract_dir = "/tmp/cnn_extract"
            z.extract(weights_name, extract_dir)
        cnn_model.load_weights(os.path.join(extract_dir, weights_name))
        return cnn_model


if "extract_fusion_embeddings" not in globals():
    def extract_fusion_embeddings(cnn_model, dataset):
        embedder = models.Model(cnn_model.input, cnn_model.get_layer("fusion_bn").output)
        embs, y_mal_list, y_obf_list = [], [], []
        for inputs, targets in dataset:
            embs.append(tf.cast(embedder(inputs, training=False), tf.float32).numpy())
            y_mal_list.append(targets["malware"].numpy())
            y_obf_list.append(targets["obfuscated"].numpy())
        return np.concatenate(embs), np.concatenate(y_mal_list), np.concatenate(y_obf_list)


if "get_predictions" not in globals():
    def get_predictions(model, dataset):
        labels_mal, labels_obf, probs_mal, probs_obf = [], [], [], []
        for inputs, targets in dataset:
            preds = model(inputs, training=False)
            labels_mal.append(targets["malware"].numpy())
            labels_obf.append(targets["obfuscated"].numpy())
            probs_mal.append(np.asarray(preds["malware"]).flatten())
            probs_obf.append(np.asarray(preds["obfuscated"]).flatten())
        return {
            "labels_mal": np.concatenate(labels_mal).flatten(),
            "probs_mal": np.concatenate(probs_mal),
            "labels_obf": np.concatenate(labels_obf).flatten(),
            "probs_obf": np.concatenate(probs_obf)
        }


if "print_classification_report" not in globals():
    def print_classification_report(res):
        print("\n--- MALWARE ---")
        print(classification_report(res["labels_mal"], (res["probs_mal"] > 0.5).astype(int),
                                     target_names=["Goodware", "Malware"]))
        print(f"ROC-AUC: {roc_auc_score(res['labels_mal'], res['probs_mal']):.4f}")
        print("\n--- OFFUSCAMENTO ---")
        print(classification_report(res["labels_obf"], (res["probs_obf"] > 0.5).astype(int),
                                     target_names=["Non offuscato", "Offuscato"]))
        print(f"ROC-AUC: {roc_auc_score(res['labels_obf'], res['probs_obf']):.4f}")


if "plot_confusion_matrix" not in globals():
    def plot_confusion_matrix(labels, preds, title, out_path, cmap="Blues"):
        cm = confusion_matrix(labels, preds)
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt="d", cmap=cmap)
        plt.title(title); plt.xlabel("Predetto"); plt.ylabel("Reale")
        _save_fig(out_path)


if "save_plots" not in globals():
    def save_plots(history, res, output_dir):
        pm, lm = np.array(res["probs_mal"]), np.array(res["labels_mal"])
        po, lo = np.array(res["probs_obf"]), np.array(res["labels_obf"])

        if history is not None and "loss" in history.history:
            plt.figure(figsize=(8, 4))
            plt.plot(history.history["loss"], label="Train")
            plt.plot(history.history["val_loss"], label="Val")
            plt.title("Loss per Epoca"); plt.xlabel("Epoca"); plt.ylabel("Loss"); plt.legend()
            _save_fig(os.path.join(output_dir, "loss_curve.png"))

        plot_confusion_matrix(lm, (pm > 0.5).astype(int), "Confusion Matrix - Malware",
                               os.path.join(output_dir, "confusion_matrix_malware.png"))
        plot_confusion_matrix(lo, (po > 0.5).astype(int), "Confusion Matrix - Offuscato",
                               os.path.join(output_dir, "confusion_matrix_obfuscation.png"), cmap="Oranges")

        df_res = pd.DataFrame({"Confidenza": pm, "Classe": lm}).replace({"Classe": {0: "Goodware", 1: "Malware"}})
        plt.figure(figsize=(8, 6))
        sns.boxplot(x="Classe", y="Confidenza", data=df_res, hue="Classe",
                    palette={"Goodware": "blue", "Malware": "red"}, legend=False)
        plt.axhline(0.5, color="black", linestyle="--")
        _save_fig(os.path.join(output_dir, "confidence_boxplot_malware.png"))

        for name, labels, probs, fname in [("Malware", lm, pm, "malware"), ("Offuscamento", lo, po, "obfuscation")]:
            fpr, tpr, _ = roc_curve(labels, probs)
            plt.figure(figsize=(6, 5))
            plt.plot(fpr, tpr, label=f"AUC={auc(fpr, tpr):.4f}")
            plt.plot([0, 1], [0, 1], "k--")
            plt.title(f"ROC - {name}"); plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.legend()
            _save_fig(os.path.join(output_dir, f"roc_curve_{fname}.png"))

            prec, rec, _ = precision_recall_curve(labels, probs)
            plt.figure(figsize=(6, 5))
            plt.plot(rec, prec)
            plt.title(f"PR Curve - {name}"); plt.xlabel("Recall"); plt.ylabel("Precision")
            _save_fig(os.path.join(output_dir, f"pr_curve_{fname}.png"))


if "evaluate_cnn_on_test" not in globals():
    def evaluate_cnn_on_test(test_ds_no_weight):
        cnn_model = load_trained_cnn(os.path.join(CNN_MODEL_DIR, "best_model.keras"))
        res = get_predictions(cnn_model, test_ds_no_weight)
        print_classification_report(res)
        save_plots(None, res, output_dir=CNN_TEST_DIR)
        return cnn_model, res


# ==============================================================
# XGBOOST — classificatore supervisionato sugli embedding
# A differenza di IF/VAE/LOF (non supervisionati), qui si usano
# le pseudo-label is_obfuscated come target di training: non è
# anomaly detection in senso stretto, ma un quinto punto di vista
# supervisionato sulla stessa rappresentazione (embedding fusion_bn).
# ==============================================================

try:
    import xgboost as xgb
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost"], check=True)
    import xgboost as xgb

XGB_DIR = os.path.join(BASE_DIR, "xgb_model")
XGB_MODEL_DIR, XGB_TRAIN_DIR, XGB_TEST_DIR = (os.path.join(XGB_DIR, s) for s in ["model", "train", "test"])
for d in [XGB_MODEL_DIR, XGB_TRAIN_DIR, XGB_TEST_DIR]:
    os.makedirs(d, exist_ok=True)

XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 5,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "eval_metric": "auc",
    "random_state": SEED,
    "n_jobs": -1,
}
XGB_EARLY_STOPPING_ROUNDS = 20


def plot_xgb_results(probs, y_obf, pct, model, output_dir):
    plt.figure(figsize=(8, 5))
    sns.histplot(probs[y_obf == 0], color="blue", label="Non offuscato", kde=True, stat="density", alpha=0.4)
    sns.histplot(probs[y_obf == 1], color="red", label="Offuscato", kde=True, stat="density", alpha=0.4)
    plt.title("Distribuzione Probabilità — XGBoost"); plt.legend()
    _save_fig(os.path.join(output_dir, "xgb_score_distribution.png"))

    df_score = pd.DataFrame({"Score": probs, "Classe": y_obf}).replace({"Classe": {0: "Non offuscato", 1: "Offuscato"}})
    plt.figure(figsize=(7, 5))
    sns.boxplot(x="Classe", y="Score", data=df_score, hue="Classe",
                palette={"Non offuscato": "blue", "Offuscato": "red"}, legend=False)
    plt.title("Probabilità per Classe — XGBoost")
    _save_fig(os.path.join(output_dir, "xgb_score_boxplot.png"))

    fpr, tpr, _ = roc_curve(y_obf, probs)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC={auc(fpr, tpr):.4f}"); plt.plot([0, 1], [0, 1], "k--")
    plt.title("ROC — XGBoost"); plt.legend()
    _save_fig(os.path.join(output_dir, "xgb_roc_curve.png"))

    prec, rec, _ = precision_recall_curve(y_obf, probs)
    plt.figure(figsize=(6, 5))
    plt.plot(rec, prec); plt.title("PR Curve — XGBoost")
    _save_fig(os.path.join(output_dir, "xgb_pr_curve.png"))

    plt.figure(figsize=(7, 5))
    plt.scatter(probs, pct, c=y_obf, cmap="coolwarm", alpha=0.5, s=10)
    plt.title("Probabilità vs Percentuale Calibrata — XGBoost"); plt.colorbar(label="is_obfuscated")
    _save_fig(os.path.join(output_dir, "xgb_score_vs_pct.png"))

    importances = model.feature_importances_
    top_idx = np.argsort(importances)[-20:]
    plt.figure(figsize=(7, 6))
    plt.barh(range(len(top_idx)), importances[top_idx])
    plt.yticks(range(len(top_idx)), [f"emb_{i}" for i in top_idx])
    plt.title("Feature Importance (Top 20 dimensioni embedding) — XGBoost")
    _save_fig(os.path.join(output_dir, "xgb_feature_importance.png"))


def train_embedding_xgb(train_emb, y_obf_train, val_emb, y_obf_val):
    t0 = time.time()
    model = xgb.XGBClassifier(**XGB_PARAMS, early_stopping_rounds=XGB_EARLY_STOPPING_ROUNDS)
    model.fit(train_emb, y_obf_train, eval_set=[(val_emb, y_obf_val)], verbose=False)
    print(f"XGBoost fit completato in {time.time() - t0:.1f}s "
          f"(best_iteration={model.best_iteration})")
    return model


def main_train_xgb():
    t_start = time.time()
    _, csv_scaler, obf_dicts = fit_and_save_scalers()
    train_ds = make_dataset("train", obf_dicts["train"], csv_scaler, shuffle=True, with_sample_weight=False)
    val_ds = make_dataset("val", obf_dicts["val"], csv_scaler, shuffle=False, with_sample_weight=False)

    best_model_path = os.path.join(CNN_MODEL_DIR, "best_model.keras")
    if not os.path.isfile(best_model_path):
        raise FileNotFoundError(f"CNN non trovata in {best_model_path}. Esegui prima la cella CNN.")
    cnn_model = load_trained_cnn(best_model_path)

    train_emb, _, y_obf_train = extract_fusion_embeddings(cnn_model, train_ds)
    val_emb, _, y_obf_val = extract_fusion_embeddings(cnn_model, val_ds)

    xgb_model = train_embedding_xgb(train_emb, y_obf_train, val_emb, y_obf_val)
    joblib.dump(xgb_model, os.path.join(XGB_MODEL_DIR, "xgb_embedding.pkl"))

    val_probs = xgb_model.predict_proba(val_emb)[:, 1]
    val_pct = val_probs * 100
    print("\n--- XGBoost (val) — OFFUSCAMENTO ---")
    print(classification_report(y_obf_val, (val_probs > 0.5).astype(int),
                                 target_names=["Non offuscato", "Offuscato"]))
    print(f"XGBoost ROC-AUC (val): {roc_auc_score(y_obf_val, val_probs):.4f}")

    plot_xgb_results(val_probs, y_obf_val, val_pct, xgb_model, XGB_TRAIN_DIR)
    print(f"Training XGBoost completato in {(time.time() - t_start) / 60:.1f} min")
    return xgb_model


def main_test_xgb():
    t_start = time.time()
    csv_scaler, obf_labeler = load_scaler_and_labeler()
    test_ds_eval = load_test_dataset(csv_scaler, obf_labeler)
    cnn_model, _ = evaluate_cnn_on_test(test_ds_eval)

    xgb_model = joblib.load(os.path.join(XGB_MODEL_DIR, "xgb_embedding.pkl"))

    test_emb, _, y_obf_test = extract_fusion_embeddings(cnn_model, test_ds_eval)
    test_probs = xgb_model.predict_proba(test_emb)[:, 1]
    test_pct = test_probs * 100

    print("\n--- XGBoost (test) — OFFUSCAMENTO ---")
    print(classification_report(y_obf_test, (test_probs > 0.5).astype(int),
                                 target_names=["Non offuscato", "Offuscato"]))
    print(f"XGBoost ROC-AUC (test): {roc_auc_score(y_obf_test, test_probs):.4f}")

    plot_xgb_results(test_probs, y_obf_test, test_pct, xgb_model, XGB_TEST_DIR)
    csv_out = save_unsupervised_predictions(test_pct, XGB_TEST_DIR, "obfuscation_percentage_test_xgb.csv")
    print(f"Test XGBoost completato in {(time.time() - t_start) / 60:.1f} min | CSV: {csv_out}")


xgb_model = main_train_xgb()
main_test_xgb()

Train: 36047 | Val: 7725 | Test: 7723
Offuscati train: 3571/36047 (9.9%)
Pesi ImageNet caricati da /content/drive/MyDrive/ProgettoSicurezzaReti/cnn_model/model/efficientnetb0_notop.h5
XGBoost fit completato in 9.2s (best_iteration=80)

--- XGBoost (val) — OFFUSCAMENTO ---
               precision    recall  f1-score   support

Non offuscato       1.00      1.00      1.00      6956
    Offuscato       0.97      0.96      0.96       756

     accuracy                           0.99      7712
    macro avg       0.98      0.98      0.98      7712
 weighted avg       0.99      0.99      0.99      7712

XGBoost ROC-AUC (val): 0.9992


/tmp/ipykernel_785/269505058.py:78: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout(); plt.savefig(path); plt.close()


Training XGBoost completato in 13.4 min
Pesi ImageNet caricati da /content/drive/MyDrive/ProgettoSicurezzaReti/cnn_model/model/efficientnetb0_notop.h5

--- MALWARE ---
              precision    recall  f1-score   support

    Goodware       1.00      1.00      1.00      5684
     Malware       1.00      1.00      1.00      2028

    accuracy                           1.00      7712
   macro avg       1.00      1.00      1.00      7712
weighted avg       1.00      1.00      1.00      7712

ROC-AUC: 1.0000

--- OFFUSCAMENTO ---
               precision    recall  f1-score   support

Non offuscato       1.00      0.99      1.00      6961
    Offuscato       0.95      0.97      0.96       751

     accuracy                           0.99      7712
    macro avg       0.98      0.98      0.98      7712
 weighted avg       0.99      0.99      0.99      7712

ROC-AUC: 0.9976

--- XGBoost (test) — OFFUSCAMENTO ---
               precision    recall  f1-score   support

Non offuscato       1.0

/tmp/ipykernel_785/269505058.py:78: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout(); plt.savefig(path); plt.close()


Test XGBoost completato in 5.7 min | CSV: /content/drive/MyDrive/ProgettoSicurezzaReti/xgb_model/test/obfuscation_percentage_test_xgb.csv
